#### 8. Design a full three-level namespace plan for Cyntexa (catalogs for dev/staging/prod, schemas per business domain) and justify the structure in a short writeup.

In [0]:
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog)
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).sales
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).purchases
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).customers
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).products
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).marketing
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).finance
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).human_resources
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).engineering
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog).operations


-- As we are designing a three-level namspacing plan for Cyntexa, these two above lines of code are enough to create the catalog and schema. As the variables 'catalog' can be set to any value at the time of deployment and we do not have the ovehead of chaning the values manually.
-- We can simply create as many environment as we want to and can deploy the same code to all of them with only adding the variables into the YAML of the project.
-- These are the main schemas that we will be usually used in business domains.

#### 9. Write a data-masking strategy: which columns need masking, which role tiers should see unmasked data, and how Unity Catalog permissions would enforce it (this connects forward to Day 8 governance).


## Data Masking Strategy for Cyntexa

### Columns Requiring Masking

**High-Sensitivity (Full Masking)**
- **Customer PII**: SSN, passport numbers, driver's license, credit card numbers, bank account numbers
- **Health Information**: Medical records, health insurance IDs
- **Authentication**: Passwords, API keys, tokens, secrets

**Medium-Sensitivity (Partial Masking)**
- **Contact Information**: Email addresses (show domain only), phone numbers (mask middle digits)
- **Financial Data**: Salary details, transaction amounts above threshold
- **Personal Identifiers**: Full names (show initials), addresses (show city/state only)

**Low-Sensitivity (Conditional Access)**
- **Business Metrics**: Revenue by customer segment, profit margins
- **Operational Data**: Customer purchase history, behavioral analytics

---

### Role-Based Access Tiers

#### Tier 1: Data Stewards & Compliance Officers (Unmasked)
- **Access**: Full unmasked data across all domains
- **Justification**: Required for compliance audits, data quality checks, and regulatory reporting
- **Examples**: Chief Data Officer, Privacy Officers, Legal team

#### Tier 2: Analytics & Data Science (Partial Masking)
- **Access**: Tokenized/hashed PII for joins, aggregated financial data, anonymized customer behavior
- **Justification**: Need to perform statistical analysis without exposing individual identities
- **Examples**: Data Scientists, Business Analysts, Marketing Analytics

#### Tier 3: Business Users (Heavily Masked)
- **Access**: Fully masked PII, aggregated metrics only, role-specific data subsets
- **Justification**: Operational needs for reporting and decision-making without individual-level access
- **Examples**: Sales Managers, Marketing Coordinators, Product Managers

#### Tier 4: External Partners & Contractors (No Access by Default)
- **Access**: Explicitly granted, time-limited, heavily masked or synthetic data only
- **Justification**: Minimum necessary principle; external parties should have minimal data exposure

---

### Unity Catalog Enforcement Strategy

#### 1. **Dynamic Views with Row Filters**
```sql
-- Apply row-level security based on user role
CREATE OR REPLACE VIEW customers.masked_customer_data AS
SELECT 
  customer_id,
  CASE 
    WHEN IS_MEMBER('data_stewards') THEN full_name
    WHEN IS_MEMBER('analytics_team') THEN CONCAT(LEFT(full_name, 1), '***')
    ELSE 'REDACTED'
  END AS full_name,
  CASE 
    WHEN IS_MEMBER('data_stewards') THEN ssn
    WHEN IS_MEMBER('analytics_team') THEN SHA2(ssn, 256) -- Hashed for joining
    ELSE NULL
  END AS ssn,
  CASE 
    WHEN IS_MEMBER('data_stewards') THEN email
    WHEN IS_MEMBER('analytics_team') THEN CONCAT('***@', SPLIT(email, '@')[1])
    ELSE NULL
  END AS email
FROM customers.raw_customer_data;
```

#### 2. **Column Masking Policies**
```sql
-- Create reusable masking function
CREATE FUNCTION customers.mask_pii(value STRING)
RETURNS STRING
RETURN CASE
  WHEN IS_MEMBER('data_stewards') THEN value
  WHEN IS_MEMBER('analytics_team') THEN SHA2(value, 256)
  ELSE 'REDACTED'
END;

-- Apply to table column
ALTER TABLE customers.customer_details
ALTER COLUMN ssn SET MASK customers.mask_pii;
```

#### 3. **Grant Structure**
```sql
-- Tier 1: Data Stewards (unmasked)
GRANT SELECT ON TABLE customers.raw_customer_data TO `data_stewards`;
GRANT USE SCHEMA ON SCHEMA customers TO `data_stewards`;

-- Tier 2: Analytics (masked views only)
GRANT SELECT ON VIEW customers.masked_customer_data TO `analytics_team`;
REVOKE SELECT ON TABLE customers.raw_customer_data FROM `analytics_team`;

-- Tier 3: Business Users (aggregated views only)
GRANT SELECT ON VIEW customers.aggregated_metrics TO `business_users`;
REVOKE SELECT ON TABLE customers.raw_customer_data FROM `business_users`;
```

#### 4. **Audit Logging**
- Enable **Unity Catalog audit logs** to track all access to sensitive columns
- Monitor `IS_MEMBER()` function usage to detect privilege escalation
- Set alerts for direct access to raw tables by non-steward roles


#### 10. (Data Analyst) Using samples.tpch, write a query with at least one CTE and one window function to produce a 'top 5 customers by revenue per region' report.

In [0]:
with customer_revenue as (
    select 
        r.r_name as region_name,
        c.c_name as customer_name,
        sum(o.o_totalprice) as total_revenue,
        row_number() over (partition by r.r_name order by sum(o.o_totalprice) desc) as revenue_rank
    from samples.tpch.customer c
    join samples.tpch.nation n on c.c_nationkey = n.n_nationkey
    join samples.tpch.region r on n.n_regionkey = r.r_regionkey
    join samples.tpch.orders o on c.c_custkey = o.o_custkey
    group by r.r_name, c.c_name
)

select 
    region_name,
    customer_name,
    total_revenue,
    revenue_rank
from customer_revenue
where revenue_rank <= 5
order by region_name, revenue_rank